```
data-mining-project/
├── data/               # Fișierul CSV brut (cerut obligatoriu la predare)
├── notebooks/          # Notebook-ul de explorare și grafice (exportat din Colab)
├── src/                # Scripturi Python curate (.py) pentru antrenare și evaluare
├── requirements.txt    # Librăriile folosite (pandas, scikit-learn, xgboost, shap)
└── README.md           # Rezumatul executiv pentru recrutori (cu graficele cheie)
```

#### **TITLE: Tech & AI Salary Determinants: Predictive Modeling in the Remote Work Era**

Choosing cols from big dataset

In [4]:
import pandas as pd
import numpy as np

In [5]:
cols_to_keep = [
    'ConvertedCompYearly',      # Ținta (Y): Salariul anual în USD
    'Country',                  # Filtru Europa + Regiune economică
    'Employment',               # Filtru doar Full-time
    'DevType',                  # Rolul (Data/AI vs Software)
    'WorkExp',                  # Ani experiență profesională (numeric)
    'YearsCode',                # Ani totali de codare (numeric)
    'RemoteWork',               # Remote / Hybrid / In-person
    'OrgSize',                  # Mărimea companiei
    'Industry',                 # Industria (FinTech, Software, etc.)
    'ICorPM',                   # Individual Contributor vs People Manager
    'EdLevel',                  # Nivel de educație
    'AISelect',                 # Utilizare unelte AI
    'AIAgents',                 # Utilizare agenți AI autonomi
    'LanguageHaveWorkedWith',   # Pentru a calcula numărul de limbaje știute
    'PlatformHaveWorkedWith'    # Pentru a crea variabila binară Cloud (AWS/Azure/GCP)
]

In [9]:
df_raw = pd.read_csv('C:\\Users\\Teodor Turcanu\\Desktop\\ai-tech-salary-prediction\\data\\survey_results_public.csv', usecols=cols_to_keep)
print(f"1. Total rows in raw global dataset: {len(df_raw):,}")

1. Total rows in raw global dataset: 49,123


In [10]:
europe_countries = [
    'United Kingdom of Great Britain and Northern Ireland', 'Germany', 'France',
    'Netherlands', 'Poland', 'Spain', 'Italy', 'Sweden', 'Switzerland', 'Belgium',
    'Austria', 'Norway', 'Denmark', 'Finland', 'Ireland', 'Portugal', 'Czech Republic',
    'Romania', 'Greece', 'Hungary', 'Ukraine', 'Bulgaria', 'Croatia', 'Slovakia',
    'Serbia', 'Lithuania', 'Slovenia', 'Estonia', 'Latvia', 'Luxembourg', 'Iceland',
    'Cyprus', 'Malta', 'Bosnia and Herzegovina', 'Albania', 'North Macedonia',
    'Republic of Moldova', 'Montenegro', 'Belarus'
]

In [11]:
df_eu = df_raw[df_raw['Country'].isin(europe_countries)].copy()
print(f"2. Total rows in EU dataset: {len(df_eu):,}")

2. Total rows in EU dataset: 16,424


In [12]:
df_eu = df_eu[df_eu['Employment'].str.contains('Employed', na=False)].copy()
print(f"3. Total rows with active employees: {len(df_eu):,}")

3. Total rows with active employees: 12,009


Cleaning target var ConvertedCompYearly

In [13]:
df_clean = df_eu.dropna(subset=['ConvertedCompYearly']).copy()
df_clean = df_clean[
    (df_clean['ConvertedCompYearly'] >= 8000) &
    (df_clean['ConvertedCompYearly'] <= 500000)
].copy()

print(f"4. Final clean rows (Europa + Angajați + Salariu valid): {len(df_clean):,}")
print("-" * 65)

4. Final clean rows (Europa + Angajați + Salariu valid): 9,101
-----------------------------------------------------------------


In [14]:
na_summary = pd.DataFrame({
    'Missing_Values': df_clean.isnull().sum(),
    'Percent_NA(%)': (df_clean.isnull().sum() / len(df_clean) * 100).round(2)
}).sort_values(by='Percent_NA(%)', ascending=False)
print("\nNew report of missing values:")
print(na_summary)


New report of missing values:
                        Missing_Values  Percent_NA(%)
PlatformHaveWorkedWith            2132          23.43
LanguageHaveWorkedWith             550           6.04
AIAgents                           457           5.02
AISelect                           194           2.13
ICorPM                             134           1.47
WorkExp                             69           0.76
RemoteWork                          33           0.36
YearsCode                           26           0.29
Industry                            12           0.13
EdLevel                              6           0.07
OrgSize                              6           0.07
DevType                              0           0.00
Employment                           0           0.00
Country                              0           0.00
ConvertedCompYearly                  0           0.00


In [15]:
print("\nDescriptive stats for ConvertedCompYearly (USD):")
print(df_clean['ConvertedCompYearly'].describe().round(2))


Descriptive stats for ConvertedCompYearly (USD):
count      9101.00
mean      82867.71
std       47796.64
min        8000.00
25%       52207.00
50%       74878.00
75%      102106.00
max      464059.00
Name: ConvertedCompYearly, dtype: float64




> *„Deoarece lucrăm cu un sondaj auto-raportat, am aplicat o filtrare de consistență economică asupra variabilei țintă: am eliminat observațiile fără salariu declarat și am restrâns intervalul la 8.000 – 500.000 USD anual. Astfel am exclus erorile tipice de raportare — precum introducerea salariului lunar în locul celui anual, confuziile de curs valutar sau valorile neserioase — care ar fi introdus heteroscedasticitate și ar fi distorsionat estimatorii OLS și metricile RMSE.”*



Data Enrichment — Data Preparation -> we add PPP

In [17]:
# mapare a indicelui nivelului prețurilor (Eurostat, EU27 = 100 -> 1.00)
# Valorile < 1.00 înseamnă cost al vieții mai mic decât media UE; > 1.00 înseamnă mai scump
price_level_index = {
    'Switzerland': 1.74, 'Iceland': 1.56, 'Norway': 1.41, 'Denmark': 1.43,
    'Ireland': 1.42, 'Luxembourg': 1.37, 'Netherlands': 1.18, 'United Kingdom of Great Britain and Northern Ireland': 1.25,
    'Sweden': 1.22, 'Finland': 1.25, 'Germany': 1.11, 'Belgium': 1.15,
    'Austria': 1.14, 'France': 1.11, 'Italy': 0.99, 'Spain': 0.92,
    'Estonia': 0.91, 'Czech Republic': 0.81, 'Slovakia': 0.83, 'Portugal': 0.86,
    'Greece': 0.83, 'Lithuania': 0.78, 'Latvia': 0.79, 'Croatia': 0.73,
    'Hungary': 0.68, 'Poland': 0.66, 'Romania': 0.60, 'Bulgaria': 0.59,
    'Serbia': 0.61, 'Ukraine': 0.45
}

df_clean['Price_Index'] = df_clean['Country'].map(price_level_index)
df_clean['Salary_PPP_Adjusted'] = (df_clean['ConvertedCompYearly'] / df_clean['Price_Index']).round(2)

print(df_clean['Salary_PPP_Adjusted'].describe().round(2))

count      8982.00
mean      78220.03
std       43059.39
min        5981.82
25%       51055.99
50%       70027.03
75%       94065.77
max      535754.10
Name: Salary_PPP_Adjusted, dtype: float64
